# Тема 15–16. LangGraph-агент как объект оценки

**Модуль IV · дополнительный пример**

Оцениваемая система тоже может быть собрана на фреймворке. Здесь строим агента на **LangGraph** (`create_agent`) с инструментом, **меняющим состояние**, и прогоняем на нём мини-eval: задачи с ожидаемым итоговым состоянием и программный грейдер. Плюс — как LangGraph даёт **наблюдаемость** (LangSmith) «из коробки».

### Что показано
1. LangGraph-агент с инструментом записи (`cancel_order`) — политику соблюдает **агент**, не инструмент.
2. Мини-eval: изоляция состояния, задачи с ожидаемым итогом, **проверка состояния среды** (не текста).
3. Наблюдаемость через LangSmith (трассировка вызовов и инструментов).

### Базовые кирпичи (кратко; подробно — в примере M2/05)

Если LangChain/LangGraph для вас новые, вот минимум, который используется ниже:
- **`ChatOpenAI(model, base_url, api_key, temperature)`** — объект модели (OpenAI-совместимый эндпоинт).
- **`@tool`** — декоратор, превращающий Python-функцию в инструмент (имя — из имени функции, описание — из докстринга, схема аргументов — из аннотаций типов). Модель сама Python не выполняет — она лишь *запрашивает* вызов; выполняет функцию агент.
- **`create_agent(model, tools, system_prompt=…)`** — готовый ReAct-агент (внутри — граф LangGraph) с `.invoke({"messages":[...]})`; `result["messages"][-1].content` — финальный ответ.

Особенность этого примера — инструмент, который **меняет внешнее состояние** (базу заказов), и оценка *результата этого изменения*; плюс параметр `system_prompt` и трассировка через **LangSmith**. Их разбираем ниже.

### Доступ к модели
Параметры — из переменных окружения: `LLM_API_KEY`, `LLM_BASE_URL`, `LLM_MODEL`.

In [1]:
import os, copy
from langchain_openai import ChatOpenAI
from langchain_core.tools import tool

llm = ChatOpenAI(model=os.environ.get("LLM_MODEL", "openai/gpt-4.1-mini"),
                 base_url=os.environ.get("LLM_BASE_URL"),
                 api_key=os.environ.get("LLM_API_KEY"), temperature=0)

# Среда с изменяемым состоянием (сбрасывается перед каждым прогоном)
_INIT = {"ORD-1001": {"status": "delivered"}, "ORD-1002": {"status": "shipped"},
         "ORD-1003": {"status": "processing"}}
ORDERS = copy.deepcopy(_INIT)
POLICY = "Отменить заказ можно только пока он не отправлен (статусы created/processing)."

@tool
def get_order(order_id: str) -> dict:
    """Информация о заказе (в т. ч. статус)."""
    return {"order_id": order_id, **ORDERS.get(order_id, {"error": "not_found"})}
@tool
def get_policy() -> str:
    """Политика отмены заказа."""
    return POLICY
@tool
def cancel_order(order_id: str) -> dict:
    """Отменить заказ (ВЫПОЛНЯЕТ отмену; соблюдение политики — на агенте)."""
    if order_id not in ORDERS:
        return {"status": "error", "reason": "not_found"}
    ORDERS[order_id]["status"] = "cancelled"
    return {"status": "ok", "order_id": order_id, "new_status": "cancelled"}

TOOLS = [get_order, get_policy, cancel_order]
print("Готово. Заказы:", {k: v["status"] for k, v in _INIT.items()})

Готово. Заказы: {'ORD-1001': 'delivered', 'ORD-1002': 'shipped', 'ORD-1003': 'processing'}


## LangGraph-агент (объект оценки)

Разберём имена из кода ниже.

- **`create_agent(llm, TOOLS, system_prompt=SYS)`** — тот же конструктор готового ReAct-агента, но с параметром **`system_prompt`**: это системная инструкция, которую агент подставляет перед каждым диалогом. Здесь она — центр примера: инструкция требует *сначала* проверить политику и статус и **отказать**, если отмена запрещена. Заметьте: инструмент `cancel_order` выполняет отмену **безусловно** (техническое действие), а *решение, звать ли его*, принимает агент по системной инструкции. Именно это решение мы и оцениваем.
- **`agent.invoke({"messages": [("user", query)]})`** — один прогон агента; на входе список сообщений (кортеж `("user", текст)`), на выходе — словарь с полной трассой в `["messages"]`, где `[-1].content` — финальный ответ пользователю.

Функция `run_task` оборачивает прогон и добавляет **изоляцию среды**: перед каждым запуском база `ORDERS` восстанавливается из эталона `_INIT` (`copy.deepcopy`), сохраняются снимки `before`/`after`. Это чистая eval-обвязка (не LangGraph), но она обязательна: без сброса состояния прогоны влияли бы друг на друга.

In [2]:
from langchain.agents import create_agent

SYS = ("Ты — ассистент поддержки магазина. Перед отменой заказа СНАЧАЛА проверь политику "
       "(get_policy) и статус заказа (get_order); если отмена запрещена политикой — вежливо "
       "откажи и НЕ вызывай cancel_order.")
agent = create_agent(llm, TOOLS, system_prompt=SYS)

def run_task(query: str) -> dict:
    """Прогон агента с изоляцией среды; возвращает финальный ответ и состояние БД до/после."""
    global ORDERS
    ORDERS = copy.deepcopy(_INIT)                      # изоляция
    before = copy.deepcopy(ORDERS)
    res = agent.invoke({"messages": [("user", query)]})
    return {"answer": res["messages"][-1].content, "before": before, "after": copy.deepcopy(ORDERS)}

r = run_task("Отмени заказ ORD-1003")
print("Ответ:", r["answer"][:120])
print("ORD-1003:", r["before"]["ORD-1003"]["status"], "->", r["after"]["ORD-1003"]["status"])

Ответ: Ваш заказ ORD-1003 успешно отменён. Если нужна будет помощь, обращайтесь!
ORD-1003: processing -> cancelled


## Мини-eval: задачи с ожидаемым итоговым состоянием

Две задачи — на действие (отмена `processing` разрешена) и **двунаправленная** (отмена `shipped` запрещена → состояние не должно меняться). Грейдер сверяет **итоговое состояние среды**, а не текст ответа.

In [3]:
import pandas as pd

TASKS = [
    {"id": "ok",     "query": "Отмени заказ ORD-1003", "expected": {"ORD-1003": "cancelled"}},
    {"id": "refuse", "query": "Отмени заказ ORD-1002", "expected": {}},   # shipped -> нельзя, без изменений
]

def grade_state(task, run) -> bool:
    exp = task["expected"]
    if not exp:                                        # refuse: изменений быть не должно
        return run["before"] == run["after"]
    return all(run["after"].get(o, {}).get("status") == s for o, s in exp.items())

rows = []
for t in TASKS:
    run = run_task(t["query"])
    rows.append({"task": t["id"], "state_pass": grade_state(t, run),
                 "итог": {o: v["status"] for o, v in run["after"].items()}})
print(pd.DataFrame(rows).to_string(index=False))

  task  state_pass                                                                       итог
    ok        True  {'ORD-1001': 'delivered', 'ORD-1002': 'shipped', 'ORD-1003': 'cancelled'}
refuse        True {'ORD-1001': 'delivered', 'ORD-1002': 'shipped', 'ORD-1003': 'processing'}


## Наблюдаемость через LangSmith

**LangSmith** — это отдельный сервис от авторов LangChain для трассировки и наблюдаемости LLM-приложений (веб-UI, где виден каждый прогон: какие сообщения ушли модели, какие инструменты вызывались с какими аргументами, сколько токенов/времени заняло). Для eval это ровно та «сквозная трасса», по которой разбирают, *почему* агент принял то или иное решение.

Ключевая деталь: агенты LangGraph трассируются в LangSmith **автоматически**, без единой строки кода — достаточно задать переменные окружения (ключ в код не вставляем):

```bash
export LANGSMITH_TRACING=true      # включает трассировку
export LANGSMITH_API_KEY=<ключ>    # доступ к вашему проекту LangSmith
export LANGSMITH_PROJECT=agent-eval  # имя проекта, куда складывать трассы
```

Как только эти переменные заданы, каждый `agent.invoke(...)` сам отправляет трассу в LangSmith — код агента не меняется (в этом смысл «из коробки»). Здесь ключ не задаём, поэтому трассировка выключена; ячейка ниже просто проверяет флаг `LANGSMITH_TRACING`.

In [4]:
print("LangSmith трассировка:", "включена" if os.environ.get("LANGSMITH_TRACING") == "true" else "выключена (ключ не задан)")

LangSmith трассировка: выключена (ключ не задан)


## Итоги

- Оцениваемый агент можно собрать на **LangGraph** (`create_agent`) — eval работает с ним так же, как с агентом на «голом» SDK.
- Оценивается **итоговое состояние среды**: политику соблюдает агент, а грейдер это проверяет (в т. ч. корректный отказ в двунаправленных задачах).
- LangGraph даёт **наблюдаемость** через LangSmith без изменения кода агента — удобно для трассировки и регрессионного eval.

### Справочник: что нового использовали (сверх примера M2/05)

| Имя | Библиотека / источник | Что делает |
|-----|----------------------|-----------|
| `create_agent(llm, tools, system_prompt=…)` | LangChain | агент с системной инструкцией; политику соблюдает **агент**, не инструмент |
| `@tool`-функция, меняющая внешнее состояние | LangChain | инструмент записи (`cancel_order`) — действие, результат которого и оценивается |
| переменные `LANGSMITH_TRACING` / `LANGSMITH_API_KEY` / `LANGSMITH_PROJECT` | LangSmith | включают автоматическую трассировку прогонов без правки кода |

Ключевая мысль: для eval важен не текст ответа, а **эффект действий агента на среду**; фреймворк лишь собирает агента и (через LangSmith) даёт трассу, а критерий успеха задаёте вы.